# Assignment 01: Exploratory Data Analysis of the New York City Airbnb Listings Dataset

**Author: Arslan Hashmi**

This assignment presents a comprehensive Exploratory Data Analysis (EDA) of the **New York City Airbnb Open Data (2019)**.

The goal of this analysis is to understand the factors influencing listing prices, explore patterns in room types and neighbourhood groups, and identify opportunities for hosts. The analysis involves data cleaning, feature engineering, univariate and multivariate visualizations, and statistical exploration using Python libraries such as Pandas, Matplotlib, and Seaborn.



## Loading the Dataset

For this assignment, we will use the real-world, publicly available **New York City Airbnb Open Data (2019)**, which contains information regarding approximately 4,900 Airbnb listings across New York City’s five boroughs, including details about hosts, room types, pricing, location, yearly availability, minimum nights and review activity.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
uploaded = files.upload()

df = pd.read_csv("new_york_airbnb_listings.csv")
df.head(10)

## Step 1: Inspecting the Data

Before choosing an appropriate chart for visualization, first we will check the shape of the dataset, identify data types for all columns, display descriptive statistical numeric data, identify null values and duplicate records and create two separate lists of numeric and categorical columns.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

print("Numeric Columns:", numeric_columns)
print("\nCategorical Columns:", categorical_columns)

### Initial Observations:

1. The columns `name`, `last_review` and `reviews_per_month` have a significant number of missing values, which indicates that some listings may not have been reviewed yet or have incomplete information.
2. There are no duplicate records in the dataset.
3. The `price` column has a wide range, suggesting the presence of outliers (very expensive luxury listings).
4. Most numerical columns show reasonable distributions after inspection.


## Step 2: Data Cleaning and Pre-Processing
In this step, missing values in the dataset are handled by dropping rows with missing names and filling missing values in review-related columns. Outliers in price and minimum nights are also treated using the IQR method and domain knowledge.

In [ ]:
df = df.dropna(subset=['name'])

df['reviews_per_month'] = df['reviews_per_month'].fillna(0.00)

df['last_review'] = df['last_review'].fillna('No review')

print("Missing values after cleaning:")
print(df.isnull().sum())

**Handling Missing Values:**

Missing values were detected in `name`, `last_review`, and `reviews_per_month` columns.

- Listings with missing `name` were dropped as they provide little analytical value.
- Missing values in `reviews_per_month` were filled with 0 (assuming no reviews).
- Missing `last_review` dates were filled with the string 'No review'.


In [ ]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound_price = max(0, Q1 - 1.5 * IQR)
upper_bound_price = Q3 + 1.5 * IQR

df = df[(df['price'] >= lower_bound_price) & (df['price'] <= upper_bound_price)]

# Cap minimum nights at a reasonable value
df = df[df['minimum_nights'] <= 30]

print("Shape after outlier removal:", df.shape)
print("Price range after cleaning:", df['price'].min(), "-", df['price'].max())

**Handling Outliers:**

The IQR (interquartile range) method was used for the `price` column. For `minimum_nights`, a reasonable maximum value of **30 nights** was applied based on domain knowledge for short-term rentals.


In [ ]:
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

**Fixing Data Types:**

The `last_review` column was converted from `object` to `datetime` format directly using Pandas.

**Justification**:

This allows for proper time-based analysis if needed later.


In [ ]:
def categorize_price(price):
    if price < 80:
        return 'Budget'
    elif 80 <= price < 200:
        return 'Mid-Range'
    else:
        return 'Premium'

df['price_category'] = df['price'].apply(categorize_price)

print(df['price_category'].value_counts())

**Feature Engineering – New Column: price_category**

A new categorical feature `price_category` was created by binning the continuous `price` variable into three meaningful groups:

- **Budget**: price < $80
- **Mid-Range**: $80 ≤ price < $200
- **Premium**: price ≥ $200

This helps in analyzing patterns across different price segments.


## Step 3: Univariate and Multivariate Analysis

In this section, we explore the distribution of key variables and relationships between them using visualizations.


### Visualization 1: Distribution of Listings Across Neighbourhood Groups

The bar plot below shows the number of Airbnb listings in each of New York City’s five boroughs.


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='neighbourhood_group', data=df, palette='viridis', order=df['neighbourhood_group'].value_counts().index)
plt.title('Number of Listings by Neighbourhood Group (Borough)')
plt.xlabel('Neighbourhood Group')
plt.ylabel('Number of Listings')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(df['neighbourhood_group'].value_counts())

**Interpretation:**

- Manhattan and Brooklyn have the highest number of listings, dominating the Airbnb market in NYC.
- Queens has a moderate number of listings.
- Bronx and Staten Island have significantly fewer listings.

This indicates that the majority of Airbnb activity is concentrated in Manhattan and Brooklyn.


### Visualization 2: Listing Price Distribution Across Different Room Types

The scatter plot generated below represents how listing prices vary across different room types.


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='room_type', y='price', hue='room_type', alpha=0.5, data=df, legend=False)
plt.title('Listing Price Variation by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Price ($)')
plt.tight_layout()
plt.show()

print(df.groupby('room_type')['price'].describe())

**Interpretation:**

- Entire home/apartment listings have a wide range of prices, including many high-priced (luxury) listings.
- Private rooms are more concentrated in the mid-price range.
- Shared rooms are generally the cheapest option.


### Visualization 3: Correlation Heatmap of Numeric Variables

The heatmap created below displays the correlation between all numeric columns and helps identify relationships with price.


In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='magma')
plt.title('Correlation Heatmap of Numeric Columns')
plt.tight_layout()
plt.show()

**Interpretation:**

- Most numeric variables have a weak correlation with price.
- The strongest (still modest) positive relationship with price is observed with latitude (slightly higher prices in northern areas of the selected range).
- Number of reviews, reviews per month, availability, and host listings count show very weak correlations with price.


### Visualization 4: Price Variation by Room Type Across Different Neighbourhood Groups

The boxplot produced below shows how price varies by room type across distinct neighbourhood groups (performing a three-variable analysis).


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='neighbourhood_group', y='price', hue='room_type', data=df)
plt.title('Price Distribution by Room Type Within Each Neighbourhood Group (Borough)')
plt.xlabel('Neighbourhood Group (Borough)')
plt.ylabel('Price ($)')
plt.legend(title='Room Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(df.groupby(['neighbourhood_group', 'room_type'])['price'].median())

**Interpretation:**

- In every neighbourhood group, entire home/apartment listings have the highest median prices.
- Private rooms have moderately higher median prices.
- Shared rooms are the cheapest.
- The price gap is most concentrated in Manhattan.

- This shows that both location and room type together heavily impact each listing price.


## Key Insights and Recommendation
Based on the exploratory data analysis of the New York City Airbnb listings, the following key insights were observed and noted:

1. Manhattan has the highest listing prices among all neighbouring groups, followed by Brooklyn. On the contrary, Bronx and Staten Island have the lowest median prices.
2. Room type strongly influences listing price, such that entire home/apartment listings are significantly more expensive than private rooms, whereas shared rooms are viewed the cheapest.
3. Most listings require only 1 to 5 minimum nights. Very few listings have high minimum stay requirements, which indicates that short-term stays are preferred by majority hosts.
4. Both Brooklyn and Manhattan dominate the market in terms of the number of listings, while Bronx and Staten Island have very limited inventory.
5. Numeric variables have a weak correlation with price. Factors including number of reviews, availability, and host listing count do not heavily affect listing price compared to location and room type.

6. **Actionable Recommendation**:
New hosts should consider listing a private room in Brooklyn or Queens with a minimum stay of 2–3 nights because these regions provide a perfect balance between competitive pricing and demand, allowing new hosts to attract visitor bookings and generate reviews more effectively before shifting towards higher-priced segments such as entire homes/apartments in Manhattan.


In [ ]:
# Save the cleaned dataset
df.to_csv("new_york_airbnb_cleaned.csv", index=False)
print("Cleaned dataset saved successfully as new_york_airbnb_cleaned.csv")
